# Advanced RAG Indexing Strategies
Most RAG systems follow a simple assumption: the text you store in the vector database is the same text you retrieve and send to the LLM. In practice, that assumption is often limiting. Retrieval and generation do not have to use the same representation of a document. Instead of embedding raw text chunks, you can embed transformed versions such as summaries, generated questions, or smaller semantic units, while still returning the original document content when it is time to generate an answer.

This article explores four indexing strategies that take advantage of this separation: standard chunk indexing, sub-chunk indexing for better precision on passages containing multiple topics, query indexing for question-answering systems with predictable user intent, and summary indexing for dense or highly structured content. Using OpenAI's text-embedding-3-small for embeddings and gpt-4o-mini for generation, we implement each approach from scratch on the same five-paragraph document, making it easy to compare their strengths, weaknesses, and retrieval performance.


## Setting up the dependencies

In [ ]:
import os
import json
import numpy as np
from openai import OpenAI
from getpass import getpass

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass('Enter OpenAI API Key: ')

Enter OpenAI API Key: ··········


In [ ]:
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
GEN_MODEL   = "gpt-4o-mini"

## Creating the Document & Utility Functions
To evaluate different RAG indexing strategies, we use a compact but information-dense technical document that covers several distinct AI concepts, including transformers, positional encodings, language model pre-training, fine-tuning with LoRA, and Retrieval-Augmented Generation (RAG). By combining multiple topics into a small set of paragraphs, the document creates realistic retrieval challenges where a query may target only a specific concept, making it easier to observe how different indexing approaches affect retrieval accuracy and relevance.

Before implementing the indexing strategies, we define a small set of utility functions that will be reused throughout the experiments. The `embed()` function converts text into vector embeddings using OpenAI's embedding model, while `cosine_sim()` computes similarity scores between query and document vectors. The `top_k_indices()` helper retrieves the highest-scoring matches, simulating a simple vector search. Finally, `paragraph_chunks()` splits the sample document into paragraph-level chunks, creating the baseline units that will be indexed, retrieved, and compared across different RAG strategies.


In [ ]:
DOCUMENT = """
Transformer models use self-attention to weigh the relevance of each token
against every other token in a sequence. This allows the model to capture
long-range dependencies that RNNs historically struggled with.

Positional encodings are added to token embeddings so the model understands
token order. Without them, attention is permutation-invariant and loses
sequence structure entirely.

During pre-training, models like BERT use masked language modeling (MLM),
where random tokens are hidden and the model must predict them. GPT-style
models use causal language modeling (CLM), predicting the next token given
all previous ones.

Fine-tuning adapts a pre-trained model to a specific task by continuing
training on a labeled dataset. LoRA (Low-Rank Adaptation) reduces the
cost of fine-tuning by injecting trainable low-rank matrices into frozen
weight layers, updating far fewer parameters than full fine-tuning.

Retrieval-Augmented Generation (RAG) grounds LLM responses in external
knowledge. A retriever fetches relevant documents at query time, which are
appended to the prompt before generation. This reduces hallucination and
keeps responses factually anchored without retraining the model.
"""

# ────────────────────────────────────────────────────────────
# Utility: embed a list of texts, return numpy matrix
# ────────────────────────────────────────────────────────────

def embed(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([r.embedding for r in response.data])

def cosine_sim(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return (a @ b.T).squeeze()

def top_k_indices(scores: np.ndarray, k: int) -> list[int]:
    return np.argsort(scores)[::-1][:k].tolist()

# ────────────────────────────────────────────────────────────
# Shared: split document into paragraph-level chunks
# ────────────────────────────────────────────────────────────

def paragraph_chunks(doc: str) -> list[str]:
    return [p.strip() for p in doc.strip().split("\n\n") if p.strip()]

CHUNKS = paragraph_chunks(DOCUMENT)

print("=" * 60)
print("DOCUMENT CHUNKS")
print("=" * 60)
for i, c in enumerate(CHUNKS):
    print(f"\n[Chunk {i}]\n{c}")

DOCUMENT CHUNKS

[Chunk 0]
Transformer models use self-attention to weigh the relevance of each token
against every other token in a sequence. This allows the model to capture
long-range dependencies that RNNs historically struggled with.

[Chunk 1]
Positional encodings are added to token embeddings so the model understands
token order. Without them, attention is permutation-invariant and loses
sequence structure entirely.

[Chunk 2]
During pre-training, models like BERT use masked language modeling (MLM),
where random tokens are hidden and the model must predict them. GPT-style
models use causal language modeling (CLM), predicting the next token given
all previous ones.

[Chunk 3]
Fine-tuning adapts a pre-trained model to a specific task by continuing
training on a labeled dataset. LoRA (Low-Rank Adaptation) reduces the
cost of fine-tuning by injecting trainable low-rank matrices into frozen
weight layers, updating far fewer parameters than full fine-tuning.

[Chunk 4]
Retrieval-Augme

## Chunk Indexing
Chunk indexing is the simplest and most common RAG approach. Each document chunk is embedded and stored in the vector database, and user queries are embedded into the same vector space at retrieval time. Similarity search is then used to identify the most relevant chunk. In this example, the query *"How does BERT learn during pre-training?"* is compared against all five paragraph embeddings. Chunk 2 achieves the highest similarity score (0.5951), significantly higher than the remaining chunks, and is therefore selected as the retrieval result. The retrieved passage directly discusses BERT's masked language modeling (MLM) objective during pre-training, demonstrating that standard chunk-level indexing works well when the information need aligns closely with a single, self-contained paragraph.


In [ ]:
print("\n\n" + "=" * 60)
print("STRATEGY 1: CHUNK INDEXING")
print("=" * 60)

chunk_embeddings = embed(CHUNKS)

query_1 = "How does BERT learn during pre-training?"
q_emb_1 = embed([query_1])
scores_1 = cosine_sim(q_emb_1, chunk_embeddings)

print(f"\nQuery: {query_1}")
print("\nSimilarity scores per chunk:")
for i, s in enumerate(scores_1):
    print(f"  Chunk {i}: {s:.4f}")

top_1 = top_k_indices(scores_1, k=1)[0]
print(f"\n→ Retrieved Chunk {top_1}:\n{CHUNKS[top_1]}")



STRATEGY 1: CHUNK INDEXING

Query: How does BERT learn during pre-training?

Similarity scores per chunk:
  Chunk 0: 0.4268
  Chunk 1: 0.3385
  Chunk 2: 0.5951
  Chunk 3: 0.3444
  Chunk 4: 0.3538

→ Retrieved Chunk 2:
During pre-training, models like BERT use masked language modeling (MLM),
where random tokens are hidden and the model must predict them. GPT-style
models use causal language modeling (CLM), predicting the next token given
all previous ones.


## Sub-chunk Indexing
Sub-chunk indexing improves retrieval precision by embedding smaller semantic units instead of entire paragraphs. Here, each paragraph is split into individual sentences, producing 11 searchable sub-chunks while preserving a mapping back to the original parent chunk. When the query *"What is positional encoding for?"* is issued, the retrieval system identifies the exact sentence discussing positional encodings with a similarity score of 0.6738.

Rather than returning only that sentence, the system returns the full parent paragraph, providing both the direct answer and its surrounding context. This strategy is particularly useful for multi-concept documents, where a large chunk may contain several topics and dilute the embedding signal, making sentence-level indexing more accurate while still preserving contextual information during generation.


In [ ]:
print("\n\n" + "=" * 60)
print("STRATEGY 2: SUB-CHUNK INDEXING")
print("=" * 60)

def sentence_split(text: str) -> list[str]:
    """Naive sentence splitter on '. ' boundaries."""
    parts = text.replace("\n", " ").split(". ")
    return [s.strip() + ("." if not s.endswith(".") else "") for s in parts if s.strip()]

# Build a flat list of (sub_chunk_text, parent_chunk_index)
sub_chunks = []
for parent_idx, chunk in enumerate(CHUNKS):
    for sentence in sentence_split(chunk):
        sub_chunks.append((sentence, parent_idx))

sub_texts   = [s for s, _ in sub_chunks]
sub_parents = [p for _, p in sub_chunks]

print(f"\nTotal sub-chunks: {len(sub_texts)}")
for i, (t, p) in enumerate(sub_chunks):
    print(f"  [{i}] parent={p} | {t[:70]}...")

sub_embeddings = embed(sub_texts)

query_2 = "What is positional encoding for?"
q_emb_2 = embed([query_2])
scores_2 = cosine_sim(q_emb_2, sub_embeddings)

print(f"\nQuery: {query_2}")
top_sub = top_k_indices(scores_2, k=1)[0]
parent_idx = sub_parents[top_sub]

print(f"\n→ Best sub-chunk [{top_sub}] (score={scores_2[top_sub]:.4f}):")
print(f"  {sub_texts[top_sub]}")
print(f"\n→ Returning full parent Chunk {parent_idx}:\n{CHUNKS[parent_idx]}")



STRATEGY 2: SUB-CHUNK INDEXING

Total sub-chunks: 11
  [0] parent=0 | Transformer models use self-attention to weigh the relevance of each t...
  [1] parent=0 | This allows the model to capture long-range dependencies that RNNs his...
  [2] parent=1 | Positional encodings are added to token embeddings so the model unders...
  [3] parent=1 | Without them, attention is permutation-invariant and loses sequence st...
  [4] parent=2 | During pre-training, models like BERT use masked language modeling (ML...
  [5] parent=2 | GPT-style models use causal language modeling (CLM), predicting the ne...
  [6] parent=3 | Fine-tuning adapts a pre-trained model to a specific task by continuin...
  [7] parent=3 | LoRA (Low-Rank Adaptation) reduces the cost of fine-tuning by injectin...
  [8] parent=4 | Retrieval-Augmented Generation (RAG) grounds LLM responses in external...
  [9] parent=4 | A retriever fetches relevant documents at query time, which are append...
  [10] parent=4 | This reduces hall

## Query Indexing
Query indexing takes a different approach: instead of embedding the document text itself, we generate and index the questions that each chunk can answer. During retrieval, the user query is matched against these hypothetical questions rather than the original content. In this example, three questions are generated for every chunk using an LLM, creating a query-oriented index.

When the user asks *"How does LoRA make fine-tuning cheaper?"*, the system finds an almost identical generated question—*"How does LoRA reduce the cost of fine-tuning?"*—with an exceptionally high similarity score of 0.9570. The corresponding parent chunk is then returned as context. This strategy often outperforms traditional chunk indexing for question-answering workloads because the indexed representations are phrased in the same form as user queries, reducing the semantic gap between how information is written and how users ask for it.


In [ ]:
print("\n\n" + "=" * 60)
print("STRATEGY 3: QUERY INDEXING")
print("=" * 60)

def generate_questions(chunk: str, n: int = 3) -> list[str]:
    prompt = (
        f"Given the following text, generate {n} concise questions that this "
        f"text directly answers. Return a JSON array of strings, nothing else.\n\n"
        f"Text:\n{chunk}"
    )
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    raw = resp.choices[0].message.content.strip()
    # Strip markdown fences if present
    raw = raw.strip("```json").strip("```").strip()
    return json.loads(raw)

# Generate questions for each chunk
print("\nGenerating hypothetical questions per chunk…")
chunk_questions = []   # list of lists
flat_questions  = []   # (question_text, chunk_index)

for i, chunk in enumerate(CHUNKS):
    qs = generate_questions(chunk, n=3)
    chunk_questions.append(qs)
    for q in qs:
        flat_questions.append((q, i))
    print(f"\n  Chunk {i} questions:")
    for q in qs:
        print(f"    • {q}")

q_texts   = [q for q, _ in flat_questions]
q_parents = [p for _, p in flat_questions]

q_embeddings = embed(q_texts)

query_3 = "How does LoRA make fine-tuning cheaper?"
q_emb_3 = embed([query_3])
scores_3 = cosine_sim(q_emb_3, q_embeddings)

print(f"\nQuery: {query_3}")
top_q = top_k_indices(scores_3, k=1)[0]
parent_idx_3 = q_parents[top_q]

print(f"\n→ Best matching question (score={scores_3[top_q]:.4f}):")
print(f"  {q_texts[top_q]}")
print(f"\n→ Returning full parent Chunk {parent_idx_3}:\n{CHUNKS[parent_idx_3]}")



STRATEGY 3: QUERY INDEXING

Generating hypothetical questions per chunk…

  Chunk 0 questions:
    • What mechanism do transformer models use to weigh the relevance of tokens?
    • What advantage do transformer models have over RNNs?
    • What type of dependencies can transformer models capture that RNNs struggled with?

  Chunk 1 questions:
    • What are positional encodings added to?
    • Why are positional encodings important for token embeddings?
    • What happens to attention without positional encodings?

  Chunk 2 questions:
    • What modeling technique do BERT models use during pre-training?
    • What is the purpose of masked language modeling (MLM) in BERT?
    • How do GPT-style models predict the next token?

  Chunk 3 questions:
    • What is the purpose of fine-tuning a pre-trained model?
    • How does LoRA reduce the cost of fine-tuning?
    • What does LoRA do to the weight layers during fine-tuning?

  Chunk 4 questions:
    • What is the purpose of Retrieval-

## Summary Indexing
Summary indexing stores embeddings of LLM-generated summaries rather than the original document text. The idea is that a concise summary often captures the core meaning of a passage more effectively than the raw text, especially when documents are long, noisy, or highly detailed. In this example, a short summary is generated for each chunk and embedded into the vector index. When the query *"How does RAG reduce hallucination?"* is issued, the summary for Chunk 4 achieves the highest similarity score (0.5145), clearly outperforming the other summaries.

The retrieved summary explicitly mentions that RAG incorporates external documents at query time to reduce hallucinations and improve factual accuracy, making it highly aligned with the user's intent. After retrieval, the system returns the original chunk rather than the summary, preserving the full context for generation. This strategy is particularly valuable when the most important concepts in a document are diluted by implementation details, examples, or other supporting information.


In [ ]:
print("\n\n" + "=" * 60)
print("STRATEGY 4: SUMMARY INDEXING")
print("=" * 60)

def summarize_chunk(chunk: str) -> str:
    prompt = (
        "Summarize the following passage in 1-2 sentences, capturing the core "
        "concepts concisely. Return only the summary.\n\n"
        f"Passage:\n{chunk}"
    )
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

print("\nGenerating summaries per chunk…")
summaries = []
for i, chunk in enumerate(CHUNKS):
    s = summarize_chunk(chunk)
    summaries.append(s)
    print(f"\n  Chunk {i} summary:\n    {s}")

summary_embeddings = embed(summaries)

query_4 = "How does RAG reduce hallucination?"
q_emb_4 = embed([query_4])
scores_4 = cosine_sim(q_emb_4, summary_embeddings)

print(f"\nQuery: {query_4}")
print("\nSimilarity scores (against summaries):")
for i, s in enumerate(scores_4):
    print(f"  Chunk {i}: {s:.4f}")

top_4 = top_k_indices(scores_4, k=1)[0]
print(f"\n→ Best summary match: Chunk {top_4} (score={scores_4[top_4]:.4f})")
print(f"  Summary: {summaries[top_4]}")
print(f"\n→ Returning full original Chunk {top_4}:\n{CHUNKS[top_4]}")



STRATEGY 4: SUMMARY INDEXING

Generating summaries per chunk…

  Chunk 0 summary:
    Transformer models utilize self-attention to assess the relevance of each token in relation to others in a sequence, enabling them to effectively capture long-range dependencies that recurrent neural networks (RNNs) have difficulty handling.

  Chunk 1 summary:
    Positional encodings are essential for maintaining the order of tokens in a model, as they are added to token embeddings; without them, the attention mechanism becomes permutation-invariant and disregards the sequence structure.

  Chunk 2 summary:
    Pre-training models like BERT utilize masked language modeling (MLM) to predict hidden tokens, while GPT-style models employ causal language modeling (CLM) to predict the next token based on prior tokens.

  Chunk 3 summary:
    Fine-tuning involves adapting a pre-trained model to a specific task using a labeled dataset, while LoRA (Low-Rank Adaptation) enhances this process by incorporatin

## Strategy Comparison
The key takeaway is that retrieval quality depends as much on what you index as on the embedding model itself. Traditional chunk indexing stores and retrieves the same text, but the other strategies demonstrate that the indexed representation can be different from the content ultimately returned to the LLM. Sub-chunk indexing improves precision by searching over smaller semantic units, query indexing aligns retrieval with how users naturally ask questions, and summary indexing focuses the embedding space on the core meaning of a passage. In all three cases, the system retrieves using a transformed representation while still returning the original source content for generation. This separation between indexing and retrieval is one of the most powerful—and often overlooked—design choices in modern RAG systems.


In [ ]:
print("\n\n" + "=" * 60)
print("STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<20} {'Whats indexed':<35} {'Whats returned'}")
print("-" * 75)
rows = [
    ("Chunk",      "Raw paragraph text",               "Same chunk"),
    ("Sub-chunk",  "Individual sentences",              "Parent paragraph"),
    ("Query",      "Hypothetical questions per chunk",  "Parent paragraph"),
    ("Summary",    "LLM-generated 1-2 sentence summary","Full original chunk"),
]
for name, indexed, returned in rows:
    print(f"{name:<20} {indexed:<35} {returned}")
print()



STRATEGY COMPARISON
Strategy             Whats indexed                       Whats returned
---------------------------------------------------------------------------
Chunk                Raw paragraph text                  Same chunk
Sub-chunk            Individual sentences                Parent paragraph
Query                Hypothetical questions per chunk    Parent paragraph
Summary              LLM-generated 1-2 sentence summary  Full original chunk

